In [0]:
%run ../00_common/data_utils

In [0]:
def calc_t_touchpoint_master(batch_id):
    # 去重处理
    touchpoint_stage_df = (spark.table(f"{get_env_config('silver_touchpoint_parsed_database')}.t_touchpoint_stage")
        .filter(F.col("BATCH_ID") == batch_id)
        .filter(F.isnotnull("tcpt_marketcode") & F.isnotnull("tcpt_brandcode") & F.isnotnull("tcpt_touchpointcode"))
        .withColumn("rn", F.row_number().over(Window.partitionBy("tcpt_marketcode", "tcpt_brandcode", "tcpt_touchpointcode", "tcpt_sourcesystemcode").orderBy(F.col("tcpt_sourcetimestamp").desc_nulls_last(),F.col("tcpt_documenttimestamp").desc_nulls_last()) 
        ))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )


    result_df = (touchpoint_stage_df
        .select(
            F.expr("uuid()").alias("tcpm_id"),
            F.col("tcpt_id").alias("tcpm_tcpt_id"),
            F.col("tcpt_action").alias("tcpm_action"),
            F.col("tcpt_documenttimestamp").alias("tcpm_documenttimestamp"),
            F.col("tcpt_documentuuid").alias("tcpm_documentuuid"),
            F.col("tcpt_recorduuid").alias("tcpm_recorduuid"),
            F.col("tcpt_sourcesystemcode").alias("tcpm_sourcesystemcode"),
            F.col("tcpt_sourcetimestamp").alias("tcpm_sourcetimestamp"),
            F.col("tcpt_marketcode").alias("tcpm_marketcode"),
            F.col("tcpt_affiliatecode").alias("tcpm_affiliatecode"),
            F.col("tcpt_divisioncode").alias("tcpm_divisioncode"),
            F.col("tcpt_brandcode").alias("tcpm_brandcode"),
            F.col("tcpt_touchpointcode").alias("tcpm_touchpointcode"),
            F.col("tcpt_submarketcode").alias("tcpm_submarketcode"),
            F.col("tcpt_auxiliarycode").alias("tcpm_auxiliarycode"),
            F.col("tcpt_auxiliarytouchpointcode").alias("tcpm_auxiliarytouchpointcode"),
            F.col("tcpt_hygieneservicecode").alias("tcpm_hygieneservicecode"),
            F.col("tcpt_distributionchannelcode").alias("tcpm_distributionchannelcode"),
            F.col("tcpt_touchpointgroupcode").alias("tcpm_touchpointgroupcode"),
            F.col("tcpt_retailerhierarchycode").alias("tcpm_retailerhierarchycode"),
            F.col("tcpt_touchpointtypecode").alias("tcpm_touchpointtypecode"),
            F.col("tcpt_englishdescription").alias("tcpm_englishdescription"),
            F.col("tcpt_localdescription").alias("tcpm_localdescription"),
            F.col("tcpt_englishfulldescription").alias("tcpm_englishfulldescription"),
            F.col("tcpt_localfulldescription").alias("tcpm_localfulldescription"),
            F.col("tcpt_descriptionen").alias("tcpm_descriptionen"),
            F.col("tcpt_descriptionlocal").alias("tcpm_descriptionlocal"),
            F.col("tcpt_fulldescriptionen").alias("tcpm_fulldescriptionen"),
            F.col("tcpt_fulldescriptionlocal").alias("tcpm_fulldescriptionlocal"),
            F.col("tcpt_url").alias("tcpm_url"),
            F.col("tcpt_jdecode").alias("tcpm_jdecode"),
            F.col("tcpt_active").alias("tcpm_active"),
            F.col("tcpt_touchpointstatus").alias("tcpm_touchpointstatus"),
            F.col("tcpt_opendate").alias("tcpm_opendate"),
            F.col("tcpt_branchid").alias("tcpm_branchid"),
            F.col("tcpt_customernumber").alias("tcpm_customernumber"),
            F.col("tcpt_redirecttouchpointcode").alias("tcpm_redirecttouchpointcode"),
            F.col("tcpt_channel").alias("tcpm_channel"),
            F.col("tcpt_customergroup").alias("tcpm_customergroup"),
            F.col("tcpt_dtcflag").alias("tcpm_dtcflag"),
            F.col("tcpt_region").alias("tcpm_region"),
            F.col("tcpt_city").alias("tcpm_city"),
            F.col("tcpt_attr_customattributelist").alias("tcpm_attr_customattributelist"),
            F.col("tcpt_phonelist").alias("tcpm_phonelist"),
            F.col("tcpt_addresslist").alias("tcpm_addresslist"),
            F.col("tcpt_customattributelist").alias("tcpm_customattributelist"),
            F.col("tcpt_terminalregistrationlist").alias("tcpm_terminalregistrationlist"),
            F.col("TCPT_GLOBAL_Level").alias("TCPM_GLOBAL_Level"),
            F.col("tcpt_global_code").alias("tcpm_global_code"),
            F.col("tcpt_global_name").alias("tcpm_global_name"),
		    F.col("tcpt_global_description").alias("tcpm_global_description"),
            F.col("TCPT_REGIONAL_Level").alias("TCPM_REGIONAL_Level"),
            F.col("tcpt_regional_code").alias("tcpm_regional_code"),
            F.col("tcpt_regional_name").alias("tcpm_regional_name"),
            F.col("tcpt_regional_description").alias("tcpm_regional_description"),
            F.col("TCPT_AFFILIATE_Level").alias("TCPM_AFFILIATE_Level"),
            F.col("tcpt_affiliate_code").alias("tcpm_affiliate_code"),
            F.col("tcpt_affiliate_name").alias("tcpm_affiliate_name"),
            F.col("tcpt_affiliate_description").alias("tcpm_affiliate_description"),
            F.current_timestamp().alias("tcpm_creation_dt"),
            F.col("tcpt_creationuid").alias("tcpm_creationuid"),
            F.current_timestamp().alias("tcpm_update_dt"),
            F.col("tcpt_updateuid").alias("tcpm_updateuid"),
            F.col("batch_id").alias("batch_id"),
            F.col("kafka_timestamp").alias("kafka_timestamp")
        )        
    )


    target_tab = DeltaTable.forName(spark, f"{get_env_config('golden_touchpoint_master_database')}.t_touchpoint_master")
    (target_tab.alias("target").merge(
        result_df.alias("source"),
         condition="""
            target.tcpm_marketcode = source.tcpm_marketcode AND
            target.tcpm_brandcode = source.tcpm_brandcode AND
            target.tcpm_touchpointcode = source.tcpm_touchpointcode AND
            target.tcpm_sourcesystemcode = source.tcpm_sourcesystemcode
        """
    )
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute()
    )

In [0]:
batch_id = dbutils.widgets.get("batch_id")
print(f"batch_id: {batch_id}")

with StepLogger("t_touchpoint_master", "05", "touchpoint", task_id=batch_id) as logger:
    calc_t_touchpoint_master(batch_id)